In [ ]:
 
from google import genai
import pandas
def ask_gemini(question):
     
    prompt = f'''You're a school system
                ,you give the user a SQL Code according to
                 what they ask for, 
                 where you compare the question with the 
                 tables that I'll give you,
                 for each question you'll recieve return 
                 SQL CODE only without any more information
                 and write the whole code in ONLY ONE CODE
                 DO NOT ADD ANY OTHER TEXT.

                 the question: {question} 
                 
                 those are my tables             
                 Courses:                        |   Department:
                 SELECT TOP (1000) [Crs_Id]      |   SELECT TOP (1000) [Dept_Id]     
                ,[Crs_Name]                      |  ,[Dept_Name]
                ,[Crs_Duration]                  |  ,[Dept_Desc]
                ,[Top_Id]                        |  ,[Dept_Location]
                FROM [school.DEPI].[dbo].[Course]|  ,[Dept_Manager]
                                                 |  ,[Manager_hiredate]
                                                 |  FROM [school.DEPI].[dbo].[Course] 
                __________________________________________________________________________
                
                Instructor:                            |  Student:
                SELECT TOP (1000) [Ins_Id]             |  SELECT TOP (1000) [St_Id]
                ,[Ins_Name]                            |  ,[St_Fname]
                ,[Ins_Degree]                          |  ,[St_Lname]
                ,[Salary]                              |  ,[St_Address]
                ,[Dept_Id]                             |  ,[St_Age]
                FROM [school.DEPI].[dbo].[Instructor]  |  ,[Dept_Id]
                                                          ,[St_super]
                                                          FROM [school.DEPI].[dbo].[Student]


            If the user asked for unknown table 
            PRINT (Can not be found)
            I DO NOT WANT ANY TEXT OR (\n)                                    
          '''

    client = genai.Client(api_key="Your_API_Key")
    
    interaction = client.interactions.create(
        model = "gemini-3.5-flash-lite",
        input = prompt
    )
    return interaction.output_text
 

In [ ]:
import pyodbc
import pandas as pd
conn = pyodbc.connect(                    
    "driver={ODBC Driver 17 for SQL Server};"
    "server=.\MSSQLSERVER02;"
    "database=school.DEPI;"
    "trusted_connection=yes;"
)
question = input('Ask gemini to get your SQL code:')
question_in_SQL_code = ask_gemini(question)
df = pd.read_sql(ask_gemini(question), conn)
df


In [22]:
question_in_SQL_code

'SELECT TOP (1000) [St_Id], [St_Fname], [St_Lname], [St_Address], [St_Age], [Dept_Id], [St_super] FROM [school.DEPI].[dbo].[Student]'

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    return df.to_html(index=False)

if __name__ == "__main__":
    app.run()

In [ ]:
from flask import Flask

app = Flask(__name__)

@app.route("/")
def home():
    table = df.to_html(index=False, classes="data")

    return f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>My Data</title>

        <style>
            body {{
                font-family: Arial;
                background-color: #f5f5f5;
                margin: 40px;
            }}

            h1 {{
                text-align: center;
                color: #333;
            }}

            .data {{
                margin: auto;
                border-collapse: collapse;
                background-color: white;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            }}

            .data th {{
                background-color: #333;
                color: white;
                padding: 12px;
            }}

            .data td {{
                padding: 10px;
                border: 1px solid #ddd;
            }}

            .data tr:hover {{
                background-color: #f1f1f1;
            }}
        </style>
    </head>

    <body>

        <h1>My DataFrame</h1>

        {table}

    </body>
    </html>
    """

if __name__ == "__main__":
    app.run()